# Vision: ball candidates in the ATD's gaps (Colab, inference only)

Runs `src/ball_detect.py` with **WASB** (Tarashima et al., BMVC 2023; [github.com/nttcom/WASB-SBDT](https://github.com/nttcom/WASB-SBDT), MIT code) and its **public soccer weights** (trained by the authors on ISSIA-CNR; licence note in `report/Build_Up_Viz.pdf`, Section 4 "Training data"). **Inference only**, with WASB's own evaluation pre- and post-processing. Nothing is trained, tuned or checked against the video.

**Rules (not negotiable):**
- **Keep Gemini disabled** in this notebook (*Settings → Colab AI*). Do not use the chat or "explain error" with data.
- **Do not look at the detections against the video**, and do not draw heatmaps or boxes on frames: checking the output against the video is forbidden.
- Only the tactical video (read from the organisers' shared folder) and `vision_bundle.zip` (segments + code) come in here. The homography and the ATD stay local.

**Tiling (`--tiles 2`).** Each frame is cut into 2 × 2 overlapping 1024 × 576 crops, and each crop goes through WASB's own preprocessing, so the ball appears ~1.9× larger. A first pass on whole frames at 512 × 288 failed the automatic check in `vision_fuse.py`: the tactical camera's ball is ~1 px there.

**Steps**
1. *Runtime → Change runtime type → GPU.* Gemini disabled.
2. Build the bundle locally with `python scripts/vision_segments.py` and upload `external/vision/vision_bundle.zip` to `MyDrive/vision/`.
3. Cells 1–3: Drive, bundle, WASB and its weights (downloaded once to Drive).
4. Cell 4: smoke test.
5. **Phase 1** (cell 5, ~30–45 min on a T4): only the 6,462 margin frames where the ATD has the ball, which is where the check runs. Cell 6 downloads `detections.csv` and `status.json` to `external/vision/out/` locally. **Stop here** until `python scripts/vision_fuse.py --check-only` prints *check passed*.
6. **Phase 2** (cell 7, several hours on a T4, less on an A100): the rest of the gaps. It resumes from phase 1 and repeats nothing. Cell 8 (optional) closes the session when it ends. Cell 9: final download to the same local folder.

**1. GPU, Drive and the video.** `VIDEO` is a shortcut to the organisers' shared tactical video at the root of My Drive.

In [ ]:
VIDEO = '/content/drive/MyDrive/BAR-ATH_tactical.mp4'
WORK = '/content/drive/MyDrive/vision'
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(WORK, exist_ok=True)
assert os.path.exists(VIDEO), 'set VIDEO'
print(os.path.getsize(VIDEO) / 1e9, 'GB')

**2. Bundle.** Replace any older `vision_bundle.zip` in Drive before running.

In [ ]:
import os, shutil
if not os.path.exists(f'{WORK}/vision_bundle.zip'):
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), f'{WORK}/vision_bundle.zip')
!rm -rf /content/vision && mkdir -p /content/vision && unzip -q {WORK}/vision_bundle.zip -d /content/vision && ls -R /content/vision
assert os.path.exists('/content/vision/segments_pads.csv'), 'old bundle: upload the new vision_bundle.zip'

**3. WASB** (MIT code) and its soccer weights: the same file its `setup_weights.sh` downloads.

In [ ]:
!pip -q install omegaconf gdown
!git clone -q https://github.com/nttcom/WASB-SBDT /content/WASB-SBDT && git -C /content/WASB-SBDT rev-parse HEAD
W = f'{WORK}/wasb_soccer_best.pth.tar'
if not os.path.exists(W):
    !gdown -q 1pg0MpMtKZ6ziYEr4oyfKYPOO3hjLw94l -O {W}
!ls -la {W}

**4. Smoke test** with tiles: 20 triplets (22 frames) end to end.

In [ ]:
%cd /content/vision
!python -m src.ball_detect --video "{VIDEO}" --segments segments.csv --wasb /content/WASB-SBDT --weights {W} --tiles 2 --out /content/smoke_tiles --smoke
!cat /content/smoke_tiles/status.json; wc -l /content/smoke_tiles/detections.csv

**5. Phase 1:** only the margins, where the ATD has the ball. Resumable: re-run this cell (and cell 3) if Colab disconnects.

In [ ]:
%cd /content/vision
!python -m src.ball_detect --video "{VIDEO}" --segments segments_pads.csv --wasb /content/WASB-SBDT --weights {W} --tiles 2 --out {WORK}/out_tiles 2>&1 | tee -a {WORK}/log_tiles.txt
!cat {WORK}/out_tiles/status.json

**6. Phase 1 download** to `external/vision/out/` locally, then run `python scripts/vision_fuse.py --check-only`.

In [ ]:
from google.colab import files
for f in ('detections.csv', 'status.json'):
    files.download(f'{WORK}/out_tiles/{f}')

**7. Phase 2**, only if the local check printed *check passed*. Same `--out`: it skips what phase 1 already did.

In [ ]:
%cd /content/vision
!python -m src.ball_detect --video "{VIDEO}" --segments segments.csv --wasb /content/WASB-SBDT --weights {W} --tiles 2 --out {WORK}/out_tiles 2>&1 | tee -a {WORK}/log_tiles.txt
!cat {WORK}/out_tiles/status.json

**8. Optional auto-shutdown.** Queue it after cell 7: it waits for Drive to sync the files, then releases the GPU. Afterwards, download `detections.csv` and `status.json` from `MyDrive/vision/out_tiles/` in the browser.

In [ ]:
import time
!tail -3 {WORK}/log_tiles.txt; cat {WORK}/out_tiles/status.json
from google.colab import drive, runtime
drive.flush_and_unmount()
time.sleep(30)
runtime.unassign()

**9. Final download** to `external/vision/out/` locally, replacing the phase 1 files.

In [ ]:
from google.colab import files
for f in ('detections.csv', 'status.json'):
    files.download(f'{WORK}/out_tiles/{f}')
files.download(f'{WORK}/log_tiles.txt')